# PC1 as a beta-hedging instrument (v2)

Can the EWMA-PC1 market mode replace SPY for beta hedging?

**The test.** Draw random long-only portfolios, hedge each one three ways, and
ask how much market-wide risk survives. Every portfolio is hedged with all three
instruments, so comparisons are paired.

| Hedge | What it is |
|---|---|
| SPY | cap-weighted ETF — the incumbent |
| RSP | equal-weighted ETF — **the control that makes the result interpretable** |
| PC1 | the eigen-weighted series from `latent_market_drift_v2` |

**Why RSP is not optional.** In the drift study PC1 correlated 0.993 with RSP.
Without RSP in the test, "PC1 beats SPY" cannot be distinguished from
"equal-weighting beats cap-weighting" — a known result with nothing to do with
principal components. The comparison worth defending is **PC1 > RSP**.

**Three hypotheses are separated and reported separately** (Cell 11):

1. PC1 hedges better than SPY.
2. PC1 removes more *systematic* risk than RSP — tested on the residual
   cross-section (Cells 5 and 6).
3. PC1 reduces more *total variance* than RSP — tested on squared residuals
   (Cell 7).

These are different questions and they do not have to agree. A hedge can remove
the common factor more completely while reducing the same total variance.

**Portfolios.** Sizes 1, 10, 20 and 50 names, 1000 draws each, under two
weighting schemes:

- **Dirichlet** — weights uniform on the simplex by stick-breaking. Concentrated
  by construction (`E[max w] = H_n/n`, about 29% for ten names), but equal in
  expectation, which is friendly terrain for PC1 and RSP alike.
- **Size-tilted** — weights proportional to a point-in-time size proxy times
  Exp(1) noise. This is a strict generalisation: normalised i.i.d. Exp(1)
  weights *are* Dirichlet(1), so with a flat proxy the two coincide and the tilt
  is the only thing that differs. It puts the test on terrain that favours SPY.

The size sweep rules out one mechanism, not the weighting effect itself. A
random portfolio converges toward the equal-weighted index as `n` grows, hence
toward PC1, and if that were driving the advantage the advantage would grow with
`n`. It does not. But an instrument closer to equal weighting also hedges a
single stock better than a capitalisation-weighted one, so a pure weighting
effect need not grow with `n` either: flatness is consistent with both readings,
and the RSP control is what separates them. Judge the sweep on the absolute
squared-residual difference, not on variance reduction, whose denominator falls
with `n` for reasons unrelated to hedging.

**Design rules, inherited.** Betas use data through `tau-1` and are applied
forward; the future-invariance test in the last cell proves it. Names are drawn
point-in-time and delisting is absorbed by renormalising across survivors, so
survivorship bias is not rebuilt inside the hedging test.

## Cell 1 — Configuration

In [ ]:
# ═══ CELL 1 ═══
import hashlib, io, json, warnings
from dataclasses import dataclass, asdict
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import requests

warnings.filterwarnings("ignore", category=FutureWarning)
warnings.filterwarnings("ignore", category=RuntimeWarning)

try:
    import yfinance as yf
except ImportError as exc:
    raise SystemExit("pip install yfinance lxml pyarrow") from exc
try:
    from IPython.display import display
except ImportError:
    display = print


@dataclass(frozen=True)
class HCfg:
    # must match latent_market_drift_v2 exactly
    start_date: str = "2015-01-01"
    end_date:   str = "2026-08-24"
    vol_half_life: int = 60
    pca_half_life: int = 252
    burn_in_days: int = 756
    vol_warmup_obs: int = 20
    min_names: int = 30
    max_gap_days: int = 4
    vol_floor: float = 1e-4
    z_clip: float = 10.0
    min_net_gross: float = 1e-3
    splice_min_cover: float = 0.20
    splice_min_days: int = 60

    # hedging study
    schemes: tuple = ("dirichlet", "tilted")
    sizes: tuple = (1, 10, 20, 50)
    n_trials: int = 1000
    beta_lookback: int = 126          # trailing window for beta
    beta_min_obs: int = 60            # usable observations required in it
    rebalance_days: int = 21          # beta refreshed monthly
    min_surviving: float = 0.5        # portfolio weight that must remain valid
    spectrum_max_k: int = 250         # portfolios entering the correlation matrix
    panel_min_cover: float = 0.90     # portfolio observation rate required
    proxy_window: int = 252           # trailing window for the size proxy
    cost_bps: float = 5.0             # per side, on hedge notional turnover
    pc1_annual_turnover: float = 1.24  # 252 x mean daily turnover, drift Cell 12
    regime_q: tuple = (0.33, 0.67)    # ex-ante vol terciles

    boot_n: int = 500                 # squared-residual bootstrap
    boot_rho_n: int = 500             # mean-correlation bootstrap
    boot_block: int = 21
    trading_days: int = 252
    seed: int = 0
    snapshot_dir: str = "./data"

CFG = HCfg()
Path(CFG.snapshot_dir).mkdir(parents=True, exist_ok=True)
FIGDIR = Path("./figures")
FIGDIR.mkdir(parents=True, exist_ok=True)


def savefig(name):
    """Write each figure to ./figures as vector PDF and raster PNG."""
    for ext in ("pdf", "png"):
        plt.savefig(FIGDIR / f"{name}.{ext}", dpi=200, bbox_inches="tight")

pd.set_option("display.width", 220)
pd.set_option("display.max_columns", 60)
HEDGES = ("SPY", "RSP", "PC1")

print(json.dumps(asdict(CFG), indent=2, default=str))

## Cell 2 — Rebuild or load the v2 state

This notebook must stand alone, so the verified `latent_market_drift_v2`
pipeline is reproduced here and cached. **Nothing about PC1 is re-derived or
re-tuned** — the same code produces the same series.

Dollar volume is also collected, as the point-in-time size proxy for the
size-tilted scheme. Market capitalisation would be preferable, but no
point-in-time cap data is available here, and today's caps applied to a 2018
portfolio would be exactly the look-ahead this project removed. Dollar volume
is observable on the day, strongly related to size, and mega-caps dominate it —
enough to move the test onto SPY's terrain. It is a proxy, and labelled as one.

In [ ]:
# ═══ CELL 2 ═══
STATE  = Path(CFG.snapshot_dir) / "hedge_state.parquet"
PANELF = Path(CFG.snapshot_dir) / "hedge_panel.parquet"
MEMBF  = Path(CFG.snapshot_dir) / "hedge_members.parquet"
VALIDF = Path(CFG.snapshot_dir) / "hedge_valid.parquet"
DVOLF  = Path(CFG.snapshot_dir) / "hedge_dollarvol.parquet"

PAGES = {"constituents": "https://en.wikipedia.org/wiki/List_of_S%26P_500_companies",
         "changes": "https://en.wikipedia.org/wiki/Historical_components_of_the_S%26P_500"}
UA = {"User-Agent": "Mozilla/5.0 (research)"}


def to_yahoo(s): return str(s).strip().upper().replace(".", "-")


def flatten_columns(cols):
    if isinstance(cols, pd.MultiIndex):
        out = []
        for c in cols:
            parts = [str(p).strip() for p in c
                     if "Unnamed" not in str(p) and str(p).strip().lower() != "nan"]
            dedup = []
            for p in parts:
                if not dedup or dedup[-1].lower() != p.lower():
                    dedup.append(p)
            out.append("_".join(dedup).lower())
        return out
    return [str(c).strip().lower() for c in cols]


def simple_returns(px, gap_days):
    px = px.astype(float); prev = px.shift(1)
    r = px / prev - 1.0
    g = px.index.to_series().diff().dt.days.to_numpy()
    ok = (g <= gap_days); ok[0] = False
    v = (px.gt(0) & prev.gt(0) & px.notna() & prev.notna()
         & pd.DataFrame(np.repeat(ok.reshape(-1, 1), px.shape[1], axis=1),
                        index=px.index, columns=px.columns))
    return r.where(v), v


def ewma_vol_lagged(R, V, hl, floor, warm):
    lam = 0.5 ** (1.0 / hl)
    r = R.to_numpy(float); m = V.to_numpy(bool)
    nt, nk = r.shape
    var = np.full(nk, np.nan); out = np.full((nt, nk), np.nan)
    cnt = np.zeros(nk, int); ready = np.zeros((nt, nk), bool)
    for t in range(nt):
        out[t] = var; ready[t] = cnt >= warm
        obs = m[t]; x = np.where(obs, r[t], 0.0) ** 2
        fresh = obs & ~np.isfinite(var)
        var = np.where(fresh, x, var)
        u = obs & ~fresh
        var = np.where(u, lam * var + (1 - lam) * x, var)
        cnt += obs
    v = np.sqrt(out)
    v = np.where(np.isfinite(v), np.maximum(v, floor), np.nan)
    return pd.DataFrame(np.where(ready, v, np.nan), index=R.index, columns=R.columns)


def run_pca(R, V, VOL, MEM, hl, min_names, z_clip, mng):
    lam = 0.5 ** (1.0 / hl)
    Rv, Vv, SG, Mv = R.to_numpy(float), V.to_numpy(bool), VOL.to_numpy(float), MEM.to_numpy(bool)
    n, nk = Rv.shape
    S = np.eye(nk); v_prev = np.zeros(nk); prev_mem = np.zeros(nk, bool)
    fac = np.full(n, np.nan)
    for t in range(n):
        ent = Mv[t] & ~prev_mem
        if ent.any():
            e = np.flatnonzero(ent); S[e, :] = 0.0; S[:, e] = 0.0; S[e, e] = 1.0
        prev_mem = Mv[t].copy()
        act = Mv[t] & Vv[t] & np.isfinite(SG[t])
        if act.sum() >= min_names:
            idx = np.flatnonzero(act)
            ev, evec = np.linalg.eigh(S[np.ix_(idx, idx)])
            v = evec[:, -1]
            ref = v_prev[idx]
            if np.any(ref):
                if np.dot(v, ref) < 0:
                    v = -v
            elif v.sum() < 0:
                v = -v
            vf = np.zeros(nk); vf[idx] = v; v_prev = vf
            raw = v / SG[t, idx]
            den = raw.sum()
            if np.isfinite(den) and abs(den) > mng * np.abs(raw).sum():
                fac[t] = float(np.dot(raw / den, Rv[t, idx]))
        z = np.zeros(nk)
        if act.any():
            z[act] = np.clip(Rv[t, act] / SG[t, act], -z_clip, z_clip)
        S = lam * S + (1 - lam) * np.outer(z, z)
    return pd.Series(fac, index=R.index, name="PC1")


def fetch_pages():
    """Raw HTML of both pages, from the snapshot the drift study wrote. Both
    parts of the study must see the same membership, and membership left live
    drifts with every Wikipedia edit."""
    snap = Path(CFG.snapshot_dir) / "wiki_membership.json"
    if snap.exists():
        blob = json.loads(snap.read_text(encoding="utf-8"))
        print(f"loaded membership snapshot {snap}  (captured {blob['captured']},"
              f" hash {blob['hash']})")
        return blob["pages"]

    session = requests.Session()
    pages = {k: session.get(u, headers=UA, timeout=60).text
             for k, u in PAGES.items()}
    digest = hashlib.sha256(
        "".join(pages[k] for k in sorted(pages)).encode("utf-8")).hexdigest()[:16]
    snap.write_text(json.dumps({
        "captured": pd.Timestamp.utcnow().strftime("%Y-%m-%d %H:%M UTC"),
        "hash": digest, "urls": PAGES, "pages": pages}), encoding="utf-8")
    print(f"wrote membership snapshot {snap}  hash {digest}")
    return pages


def build_state():
    found = []
    for page, html in fetch_pages().items():
        for hdr in (0, [0, 1]):
            try:
                for i, t in enumerate(pd.read_html(io.StringIO(html), header=hdr)):
                    found.append((f"{page} header={hdr} idx={i}", t))
            except Exception as exc:
                print(f"  read_html({page}, {hdr}) failed: {exc}")

    current = changes = None
    for label, t in found:
        cols = flatten_columns(t.columns)
        if (current is None and len(t) > 100
                and any(c in ("symbol", "ticker") for c in cols)
                and any("security" in c for c in cols)):
            current = t
        ok = (any(c.startswith("added") for c in cols)
              and any(c.startswith("removed") for c in cols)
              and any("date" in c for c in cols) and len(t) > 50)
        better = changes is None or (
            any("ticker" in c for c in cols)
            and not any("ticker" in c for c in flatten_columns(changes.columns)))
        if ok and better:
            changes = t
    if current is None or changes is None:
        for label, t in found:
            print(f"  {label:34s} rows={len(t):4d} cols={flatten_columns(t.columns)[:6]}")
        raise RuntimeError("could not locate the Wikipedia tables")

    cdf = current.copy(); cdf.columns = flatten_columns(cdf.columns)
    sym = next(c for c in cdf.columns if c in ("symbol", "ticker"))
    cur = pd.DataFrame({"ticker": cdf[sym].map(to_yahoo)}).drop_duplicates("ticker")

    ch = changes.copy(); ch.columns = flatten_columns(ch.columns)
    c_date = next(c for c in ch.columns if "date" in c)
    ac = [c for c in ch.columns if c.startswith("added")]
    rc = [c for c in ch.columns if c.startswith("removed")]
    c_add = next((c for c in ac if "ticker" in c), ac[0])
    c_rem = next((c for c in rc if "ticker" in c), rc[0])
    dts = pd.to_datetime(ch[c_date], format="mixed", errors="coerce")

    def clean(s):
        s = s.astype("string").str.strip()
        return s.where(s.notna() & (s != "")
                       & (~s.str.lower().isin(["nan", "ticker", "none", "—", "-"])))

    chg = pd.DataFrame({
        "date": dts,
        "added": clean(ch[c_add]).map(lambda x: to_yahoo(x) if pd.notna(x) else pd.NA),
        "removed": clean(ch[c_rem]).map(lambda x: to_yahoo(x) if pd.notna(x) else pd.NA),
    }).dropna(subset=["date"]).reset_index(drop=True)

    as_of = pd.Timestamp(CFG.end_date)
    cursor, timeline = set(cur.ticker), []
    for date, block in sorted(chg.groupby("date"), key=lambda kv: kv[0], reverse=True):
        # A change after as_of is UNDONE, never skipped: the cursor starts from
        # the current constituent list, which already reflects it, so skipping
        # would carry a later addition back across the whole sample and drop a
        # later removal from it. Only changes at or before as_of are recorded.
        if date <= as_of:
            timeline.append((date, frozenset(cursor)))
        prior = set(cursor)
        for tk in block["added"].dropna():
            prior.discard(tk)
        for tk in block["removed"].dropna():
            prior.add(tk)
        cursor = prior
    timeline.append((pd.Timestamp("1900-01-01"), frozenset(cursor)))
    timeline.reverse()

    ws, we = pd.Timestamp(CFG.start_date), as_of
    uni = set()
    for i, (s0, mem) in enumerate(timeline):
        s1 = timeline[i + 1][0] if i + 1 < len(timeline) else we
        if s1 >= ws and s0 <= we:
            uni |= mem
    universe = sorted(uni)
    print(f"tickers ever a member: {len(universe)}")

    snap = Path(CFG.snapshot_dir) / f"prices_{CFG.start_date}_{CFG.end_date}.parquet"
    dsnap = Path(CFG.snapshot_dir) / f"dvol_{CFG.start_date}_{CFG.end_date}.parquet"
    # The price snapshot is shared with latent_market_drift_v2 and is NEVER
    # overwritten once it exists. Re-downloading is not reproducible: the same
    # universe requested on two days returned 121 and then 134 failures, which
    # would silently give the two notebooks different universes and therefore
    # different PC1 series. If only the dollar-volume cache is missing, volume
    # is fetched for exactly the tickers already in the price snapshot.
    if snap.exists() and not dsnap.exists():
        import time
        prices = pd.read_parquet(snap)
        cols = list(prices.columns)
        print(f"price snapshot found ({len(cols)} tickers); fetching volume only")
        vframes = []
        for i in range(0, len(cols), 100):
            batch = cols[i:i + 100]
            print(f"  {i+1:4d}-{i+len(batch):4d} of {len(cols)}", flush=True)
            raw = None
            for a in range(3):
                try:
                    raw = yf.download(batch, start=CFG.start_date, end=CFG.end_date,
                                      auto_adjust=True, progress=False, threads=True)
                    if raw is not None and not raw.empty:
                        break
                except Exception as exc:
                    print(f"     retry {a+1}: {exc}")
                time.sleep(2 ** a)
            if raw is None or raw.empty:
                continue
            if isinstance(raw.columns, pd.MultiIndex):
                cl, vo = raw["Close"], raw["Volume"]
            else:
                cl, vo = raw[["Close"]], raw[["Volume"]]
                cl.columns = batch[:1]; vo.columns = batch[:1]
            keep_b = [c for c in vo.columns if c in cols]
            if keep_b:
                vframes.append(cl[keep_b] * vo[keep_b])
        dvol = pd.concat(vframes, axis=1).sort_index()
        dvol = dvol.loc[~dvol.index.duplicated()]
        dvol = dvol.reindex(columns=cols)          # keep the snapshot's universe
        dvol.to_parquet(dsnap)
        print(f"wrote dollar-volume snapshot for {dvol.notna().any().sum()} tickers")
    elif snap.exists() and dsnap.exists():
        prices = pd.read_parquet(snap); dvol = pd.read_parquet(dsnap)
        print("loaded price and dollar-volume snapshots")
    else:
        import time
        cframes, dframes, failed = [], [], []
        tick = universe + ["SPY", "RSP"]
        for i in range(0, len(tick), 100):
            batch = tick[i:i + 100]
            print(f"  {i+1:4d}-{i+len(batch):4d} of {len(tick)}", flush=True)
            raw = None
            for a in range(3):
                try:
                    raw = yf.download(batch, start=CFG.start_date, end=CFG.end_date,
                                      auto_adjust=True, progress=False, threads=True)
                    if raw is not None and not raw.empty:
                        break
                except Exception as exc:
                    print(f"     retry {a+1}: {exc}")
                time.sleep(2 ** a)
            if raw is None or raw.empty:
                failed += batch; continue
            if isinstance(raw.columns, pd.MultiIndex):
                cl, vo = raw["Close"], raw["Volume"]
            else:
                cl, vo = raw[["Close"]], raw[["Volume"]]
                cl.columns = batch[:1]; vo.columns = batch[:1]
            got = [c for c in cl.columns if cl[c].notna().any()]
            failed += sorted(set(batch) - set(got))
            if got:
                cframes.append(cl[got])
                dframes.append(cl[got] * vo[got])
        prices = pd.concat(cframes, axis=1).sort_index()
        prices = prices.loc[~prices.index.duplicated()]
        dvol = pd.concat(dframes, axis=1).sort_index()
        dvol = dvol.loc[~dvol.index.duplicated()]
        prices.to_parquet(snap); dvol.to_parquet(dsnap)
        print(f"failed downloads: {len(failed)}")

    for df in (prices, dvol):
        df.index = pd.to_datetime(df.index)
        if getattr(df.index, "tz", None) is not None:
            df.index = df.index.tz_localize(None)

    # Restricted to the universe, not to everything in the snapshot: the
    # snapshot is additive across runs, so a column downloaded for an earlier
    # universe can survive in it and would otherwise enter the panel.
    stock_cols = [c for c in prices.columns
                  if c not in ("SPY", "RSP") and c in uni]

    ev = {}
    for _, row in chg.dropna(subset=["date"]).sort_values("date").iterrows():
        if pd.notna(row["added"]):
            ev.setdefault(str(row["added"]), []).append((row["date"], "add"))
        if pd.notna(row["removed"]):
            ev.setdefault(str(row["removed"]), []).append((row["date"], "rem"))
    gaps = {}
    for tk, e in ev.items():
        g, opn = [], None
        for d, k in sorted(e, key=lambda x: (x[0], x[1])):
            if k == "rem":
                opn = d
            elif opn is not None and d > opn:
                g.append((opn, d)); opn = None
        if g:
            gaps[tk] = g
    spliced = set()
    for tk, spans in gaps.items():
        if tk not in stock_cols:
            continue
        s = prices[tk]
        if not s.notna().any():
            continue
        for lo, hi in spans:
            w = s.loc[(s.index > lo) & (s.index < hi)]
            if len(w) >= CFG.splice_min_days and float(w.notna().mean()) < CFG.splice_min_cover:
                spliced.add(tk); break
    keep = [c for c in stock_cols if c not in spliced]
    print(f"spliced dropped: {sorted(spliced)}   retained: {len(keep)}")

    rets, valid = simple_returns(prices[keep], CFG.max_gap_days)
    bench, _ = simple_returns(prices[["SPY", "RSP"]], CFG.max_gap_days)
    dates = rets.index

    col = {t: i for i, t in enumerate(keep)}
    starts = np.array([d.value for d, _ in timeline])
    M = np.zeros((len(dates), len(keep)), dtype=bool)
    for r_i, dt in enumerate(dates):
        i = max(np.searchsorted(starts, dt.value, side="right") - 1, 0)
        for tk in timeline[i][1]:
            j = col.get(tk)
            if j is not None:
                M[r_i, j] = True
    members = pd.DataFrame(M, index=dates, columns=keep)

    vol = ewma_vol_lagged(rets, valid, CFG.vol_half_life, CFG.vol_floor, CFG.vol_warmup_obs)
    print("running streaming PCA (a few minutes) ...", flush=True)
    pc1 = run_pca(rets, valid, vol, members, CFG.pca_half_life,
                  CFG.min_names, CFG.z_clip, CFG.min_net_gross)

    panel = pd.DataFrame({"PC1": pc1, "SPY": bench["SPY"], "RSP": bench["RSP"]})
    dv = dvol[keep].reindex(dates)

    rets.to_parquet(STATE); panel.to_parquet(PANELF)
    members.to_parquet(MEMBF); valid.to_parquet(VALIDF); dv.to_parquet(DVOLF)
    return rets, valid, members, panel, dv


if all(f.exists() for f in (STATE, PANELF, MEMBF, VALIDF, DVOLF)):
    rets = pd.read_parquet(STATE); valid = pd.read_parquet(VALIDF)
    members = pd.read_parquet(MEMBF); panel = pd.read_parquet(PANELF)
    dvol_df = pd.read_parquet(DVOLF)
    print("loaded cached state")
else:
    rets, valid, members, panel, dvol_df = build_state()

DATES_ALL = rets.index
burn_cut = DATES_ALL[min(CFG.burn_in_days, len(DATES_ALL) - 1)]
WIN = panel.dropna().index
WIN = WIN[WIN >= burn_cut]

rets_w  = rets.loc[WIN]
valid_w = valid.loc[WIN].fillna(False).astype(bool)
memb_w  = members.loc[WIN].astype(bool)
hedge_w = panel.loc[WIN]

# Size proxy: trailing median dollar volume, known at the draw date.
t0 = WIN[0]
hist = dvol_df.loc[dvol_df.index < t0].tail(CFG.proxy_window)
PROXY = hist.median(axis=0).reindex(rets_w.columns).to_numpy(dtype=float)

print(f"\nanalysis window : {WIN.min():%Y-%m-%d} to {WIN.max():%Y-%m-%d}  ({len(WIN):,} days)")
print(f"stocks          : {rets_w.shape[1]}")
print(f"size proxy      : trailing {CFG.proxy_window}d median dollar volume to "
      f"{hist.index.max():%Y-%m-%d}, {np.isfinite(PROXY).sum()} names priced")
print("\nhedge instrument correlations:")
print(hedge_w.corr().round(4).to_string())
print("\nannualised vol:")
print((hedge_w.std() * np.sqrt(CFG.trading_days)).map("{:.2%}".format).to_string())

## Cell 3 — Samplers

Names are drawn uniformly from index members with valid data **on the draw
date**, identically for both schemes, so weighting is the only thing that
changes between them. Delisting is absorbed by renormalising across survivors,
and a portfolio whose surviving weight falls below `min_surviving` is marked
missing rather than rescaled from a stub.

In [ ]:
# ═══ CELL 3 ═══
def stick_breaking_weights(n, size, rng):
    """Dirichlet(1,...,1) by uniform spacings: sort n-1 uniforms, take the gaps."""
    if n == 1:
        return np.ones((size, 1))
    u = rng.random((size, n - 1)); u.sort(axis=1)
    edges = np.concatenate([np.zeros((size, 1)), u, np.ones((size, 1))], axis=1)
    return np.diff(edges, axis=1)


def tilted_weights(proxy, rng):
    """w propto proxy * Exp(1). With a flat proxy this IS Dirichlet(1)."""
    proxy = np.asarray(proxy, dtype=float)
    g = rng.exponential(1.0, size=proxy.shape)
    w = np.where(np.isfinite(proxy) & (proxy > 0), proxy, 0.0) * g
    tot = w.sum(axis=1, keepdims=True)
    return np.divide(w, tot, out=np.full_like(w, np.nan), where=tot > 0)


def draw_names(eligible_idx, n, n_trials, rng):
    picks = np.empty((n_trials, n), dtype=np.int64)
    for k in range(n_trials):
        picks[k] = rng.choice(eligible_idx, size=n, replace=False)
    return picks


def weight_matrix(picks, weights, n_cols):
    W = np.zeros((picks.shape[0], n_cols))
    rows = np.repeat(np.arange(picks.shape[0]), picks.shape[1])
    W[rows, picks.ravel()] = weights.ravel()
    return W


def portfolio_returns(W, R, V, min_surviving):
    Rf = np.where(V, np.nan_to_num(R), 0.0)
    M = V.astype(float)
    num = Rf @ W.T
    den = M @ W.T
    with np.errstate(invalid="ignore", divide="ignore"):
        r_P = np.where(den > 0, num / den, np.nan)
    return np.where(den >= min_surviving, r_P, np.nan), den


Rv, Vv, Mv = rets_w.to_numpy(float), valid_w.to_numpy(bool), memb_w.to_numpy(bool)
ELIGIBLE = np.flatnonzero(Mv[0] & Vv[0] & np.isfinite(PROXY) & (PROXY > 0))
print(f"eligible names on {WIN.min():%Y-%m-%d}: {len(ELIGIBLE)}")
print(f"size proxy across eligible names: "
      f"median {np.median(PROXY[ELIGIBLE]):,.0f}, "
      f"p99/p50 = {np.percentile(PROXY[ELIGIBLE], 99)/np.median(PROXY[ELIGIBLE]):.1f}x")

## Cell 4 — Run the experiment

Betas use `Cov(r_P, r_H) = sum_i w_i Cov(r_i, r_H)`, so the per-stock rolling
covariances are computed once and every portfolio's beta is a weighted sum. Eight
thousand portfolios then cost about as much as one.

The beta in force on day `t` was estimated through `tau-1`, where `tau <= t` is
the last rebalance, so it is fixed before `r_t` is read.

Each scheme-size combination is processed and then released: holding every
residual panel at once would need several hundred megabytes, so the loop keeps
only the summaries and the aligned panels that Cell 6 resamples.

In [ ]:
# ═══ CELL 4 ═══
COV, VAR = {}, {}
for h in HEDGES:
    COV[h] = rets_w.rolling(CFG.beta_lookback, min_periods=CFG.beta_min_obs
                            ).cov(hedge_w[h]).to_numpy()
    VAR[h] = hedge_w[h].rolling(CFG.beta_lookback, min_periods=CFG.beta_min_obs
                                ).var().to_numpy()
RB = np.arange(CFG.beta_lookback, len(WIN), CFG.rebalance_days)
print(f"rolling covariances done; rebalance dates: {len(RB)}")


def hedge_betas(W, V, cov, var, rb_idx, n_days, min_surviving):
    beta = np.full((n_days, W.shape[0]), np.nan)
    for tau in rb_idx:
        src = tau - 1
        if src < 0 or not np.isfinite(var[src]) or var[src] <= 0:
            continue
        c = cov[src]; ok = np.isfinite(c)
        Wm = W * (V[src] & ok)[None, :]
        den = Wm.sum(axis=1)
        with np.errstate(invalid="ignore", divide="ignore"):
            b = np.where(den >= min_surviving,
                         (Wm @ np.nan_to_num(c)) / np.where(den > 0, den, np.nan) / var[src],
                         np.nan)
        nxt = rb_idx[rb_idx > tau].min() if (rb_idx > tau).any() else n_days
        beta[tau:nxt, :] = b[None, :]
    return beta


def variance_reduction(r_P, eps):
    out = np.full(r_P.shape[1], np.nan)
    for k in range(r_P.shape[1]):
        m = np.isfinite(r_P[:, k]) & np.isfinite(eps[:, k])
        if m.sum() < 50:
            continue
        v0 = np.var(r_P[m, k])
        if v0 > 0:
            out[k] = 1.0 - np.var(eps[m, k]) / v0
    return out


def residual_common_factor(panel_arr, max_k=None, seed=0, min_cover=0.90):
    """Drop thin portfolios first, then incomplete days.

    Requiring all K portfolios to be observed on a day is far too strict once K
    is large: a one-name portfolio is missing whenever its single stock is, so
    with 250 columns almost no day survives. Without dropping the all-missing
    warm-up first, the coverage threshold would also depend on sample length.
    """
    X = np.asarray(panel_arr, dtype=float)
    X = X[np.isfinite(X).any(axis=1)]
    cover = np.isfinite(X).mean(axis=0)
    X = X[:, cover >= min_cover]
    if X.shape[1] < 3:
        return dict(mean_corr=np.nan, lam1=np.nan, mp_edge=np.nan, T=0, K=int(X.shape[1]))
    if max_k is not None and X.shape[1] > max_k:
        X = X[:, np.sort(np.random.default_rng(seed).choice(X.shape[1], max_k, replace=False))]
    X = X[np.isfinite(X).all(axis=1)]
    T, K = X.shape
    if T < 30:
        return dict(mean_corr=np.nan, lam1=np.nan, mp_edge=np.nan, T=T, K=K)
    C = np.corrcoef(X, rowvar=False)
    ev = np.linalg.eigvalsh(np.nan_to_num(C, nan=0.0))
    q = K / T
    return dict(mean_corr=float((C.sum() - np.trace(C)) / (K * (K - 1))),
                lam1=float(ev[-1]), mp_edge=float((1 + np.sqrt(q)) ** 2), T=T, K=K)


def overlap_null(W, R, V, seed, max_k, min_surviving, min_cover,
                 row_mask=None):
    """The correlation floor created by shared holdings alone.

    Each stock's series is shuffled in time with its OWN permutation. That
    destroys every cross-sectional factor while leaving the holdings structure
    intact: two portfolios holding stock i still hold the same shuffled series.
    What survives is exactly the overlap contribution, which is the level a
    hedged panel should be read against -- not zero.
    """
    rng_l = np.random.default_rng(seed)
    Rs, Vs = R.copy(), V.copy()
    for j in range(Rs.shape[1]):
        p = rng_l.permutation(Rs.shape[0])
        Rs[:, j] = Rs[p, j]; Vs[:, j] = Vs[p, j]
    pan, _ = portfolio_returns(W, Rs, Vs, min_surviving)
    if row_mask is not None:            # same days as the hedged panels
        pan = pan[row_mask]
    return residual_common_factor(pan, max_k, seed, min_cover)


def align_panels(panels, max_k, min_cover, seed):
    """Put every instrument on the SAME portfolios and days, so pairs are paired."""
    keys = list(panels)
    arrs = [np.asarray(panels[k], dtype=float) for k in keys]
    observed = np.ones(arrs[0].shape[0], dtype=bool)
    for A in arrs:
        observed &= np.isfinite(A).any(axis=1)
    arrs = [A[observed] for A in arrs]
    cover = np.ones(arrs[0].shape[1], dtype=bool)
    for A in arrs:
        cover &= np.isfinite(A).mean(axis=0) >= min_cover
    idx = np.flatnonzero(cover)
    if max_k is not None and len(idx) > max_k:
        idx = np.sort(np.random.default_rng(seed).choice(idx, max_k, replace=False))
    arrs = [A[:, idx] for A in arrs]
    complete = np.ones(arrs[0].shape[0], dtype=bool)
    for A in arrs:
        complete &= np.isfinite(A).all(axis=1)
    return {k: A[complete] for k, A in zip(keys, arrs)}, int(complete.sum()), int(len(idx))


def spanning(eps, other, min_obs=100):
    b = np.full(eps.shape[1], np.nan)
    for k in range(eps.shape[1]):
        m = np.isfinite(eps[:, k]) & np.isfinite(other)
        if m.sum() < min_obs:
            continue
        x, y = other[m], eps[m, k]
        xc = x - x.mean(); vx = (xc ** 2).sum()
        if vx > 0:
            b[k] = float((xc * (y - y.mean())).sum() / vx)
    return np.nanmean(b)


# ex-ante volatility regimes, from SPY's lagged EWMA sigma
lam_v = 0.5 ** (1 / CFG.vol_half_life)
spy_r = hedge_w["SPY"].to_numpy()
_ev = np.full(len(WIN), np.nan); _st = np.nan
for t in range(len(WIN)):
    _ev[t] = _st
    x = spy_r[t] ** 2
    _st = x if not np.isfinite(_st) else lam_v * _st + (1 - lam_v) * x
exante = pd.Series(np.sqrt(_ev), index=WIN, name="exante_vol")
q1, q2 = exante.quantile(CFG.regime_q[0]), exante.quantile(CFG.regime_q[1])
REGIME = pd.Series(np.where(exante <= q1, "calm",
                   np.where(exante <= q2, "normal", "stressed")), index=WIN)

RESULTS, ALIGNED = {}, {}

# Names are drawn ONCE per size and reused by every scheme, so weighting is the
# only thing that differs between them. A single advancing generator shared
# across the scheme loop would not do this: by the time the second scheme runs,
# the generator sits wherever the first one left it, and the two schemes would
# silently draw different stocks. The tell is n = 1, where both schemes assign
# weight 1 and must therefore agree exactly -- asserted in Cell 12.
for n in CFG.sizes:
    name_rng = np.random.default_rng([CFG.seed, 1, n])
    picks = draw_names(ELIGIBLE, n, CFG.n_trials, name_rng)

    for si, scheme in enumerate(CFG.schemes):
        w_rng = np.random.default_rng([CFG.seed, 2, n, si])
        if scheme == "dirichlet":
            w = stick_breaking_weights(n, CFG.n_trials, w_rng)
        else:
            w = tilted_weights(PROXY[picks], w_rng)
        W = weight_matrix(picks, w, rets_w.shape[1])
        r_P, _ = portfolio_returns(W, Rv, Vv, CFG.min_surviving)

        eps, beta, vr = {}, {}, {}
        for h in HEDGES:
            beta[h] = hedge_betas(W, Vv, COV[h], VAR[h], RB, len(WIN), CFG.min_surviving)
            eps[h] = r_P - beta[h] * hedge_w[h].to_numpy()[:, None]
            vr[h] = variance_reduction(r_P, eps[h])

        key = (scheme, n)
        # Every row of the residual-cross-section tables must describe the
        # same days. Hedge ratios need 126 days of history, so the residual
        # panels start in July 2018; the unhedged and overlap rows are
        # restricted to that grid rather than running the full window.
        HG = np.isfinite(eps["PC1"]).any(axis=1)

        cf = {"unhedged": residual_common_factor(r_P[HG], CFG.spectrum_max_k,
                                                 CFG.seed, CFG.panel_min_cover),
              "overlap_null": overlap_null(W, Rv, Vv, CFG.seed, CFG.spectrum_max_k,
                                           CFG.min_surviving, CFG.panel_min_cover,
                                           row_mask=HG)}
        for h in HEDGES:
            cf[h] = residual_common_factor(eps[h], CFG.spectrum_max_k, CFG.seed,
                                           CFG.panel_min_cover)

        al, Tn, Kn = align_panels({h: eps[h] for h in HEDGES},
                                  CFG.spectrum_max_k, CFG.panel_min_cover, CFG.seed)
        ALIGNED[key] = al

        sq = {f"{a}-{b}": np.nanmean(eps[a] ** 2 - eps[b] ** 2, axis=1)
              for a, b in [("PC1", "SPY"), ("PC1", "RSP")]}

        cuts = {}
        # WIN.year == y is already an ndarray (DatetimeIndex comparison);
        # REGIME == r is a Series. np.asarray normalises both.
        for label, m in ([(f"year {y}", np.asarray(WIN.year == y))
                          for y in sorted(set(WIN.year))]
                         + [(r, np.asarray(REGIME == r))
                            for r in ("calm", "normal", "stressed")]):
            if m.sum() < 60:
                continue
            # the same per-portfolio statistic Table 12 reports, not a
            # pooled variance ratio; variance_reduction masks each portfolio
            # to the days where both its return and its residual exist, so
            # the 2018 cut can no longer divide a half-year residual variance
            # by a full-year portfolio variance
            cuts[label] = {h: float(np.nanmean(variance_reduction(r_P[m], eps[h][m])))
                           for h in HEDGES}

        span = {f"resid({a}) on {b}": spanning(eps[a], hedge_w[b].to_numpy())
                for a, b in [("PC1", "SPY"), ("PC1", "RSP"), ("SPY", "PC1"), ("RSP", "PC1")]}

        # Costs are charged on rebalancing, so only a move between two
        # betas that both exist counts. Filling the missing warm-up with
        # zeros would charge the first finite beta as a jump from nothing,
        # which is the one-off cost of putting the hedge on rather than a
        # rebalance. The charge is annualised over the days the hedge is
        # actually held, not over the full window.
        years = float(HG.sum()) / CFG.trading_days
        c = CFG.cost_bps / 1e4
        cost = {}
        for h in HEDGES:
            _b = beta[h]
            _live = np.isfinite(_b[1:]) & np.isfinite(_b[:-1])
            db = np.where(_live, np.abs(np.diff(_b, axis=0)), 0.0).sum(axis=0)
            internal = CFG.pc1_annual_turnover * c if h == "PC1" else 0.0
            cost[h] = float(np.nanmean(db) * c / years + internal)

        RESULTS[key] = dict(cf=cf, vr={h: float(np.nanmean(vr[h])) for h in HEDGES},
                            vr_p10={h: float(np.nanpercentile(vr[h], 10)) for h in HEDGES},
                            vr_p90={h: float(np.nanpercentile(vr[h], 90)) for h in HEDGES},
                            sq=sq, cuts=cuts, span=span, cost=cost,
                            beta={h: float(np.nanmean(beta[h])) for h in HEDGES},
                            maxw=float(np.nanmean(w.max(axis=1))), T=Tn, K=Kn,
                            usable=float(np.isfinite(r_P).mean()))
        print(f"  {scheme:10s} n={n:3d}: E[max w] {RESULTS[key]['maxw']:.3f}  "
              f"panel {Kn}x{Tn}  VR " +
              " ".join(f"{h} {RESULTS[key]['vr'][h]:.3f}" for h in HEDGES), flush=True)
        del eps, beta, vr, r_P, W
print("\ndone.")

# Persist the summary so every number reported in the paper can be traced to a
# run without recomputing the experiment.
import json as _json
_dump = {}
for (sch, nn), r in RESULTS.items():
    _dump[f"{sch}|{nn}"] = {
        "vr": r["vr"], "vr_p10": r["vr_p10"], "vr_p90": r["vr_p90"],
        "cf": {k: {kk: (None if not np.isfinite(vv) else float(vv))
                   for kk, vv in v.items()} for k, v in r["cf"].items()},
        "span": r["span"], "cost": r["cost"], "beta": r["beta"],
        "maxw": r["maxw"], "cuts": r["cuts"],
    }
Path(CFG.snapshot_dir, "hedge_results.json").write_text(
    _json.dumps(_dump, indent=1), encoding="utf-8")
print("wrote data/hedge_results.json")

## Cell 5 — Residual common factor

Residual variance alone cannot say whether *systematic* risk was removed: it
does not separate systematic from idiosyncratic risk, and any measure defined
against SPY or PC1 favours that instrument.

The cross-section can. Portfolios holding different random names share only
idiosyncratic noise after a perfect hedge, so their residuals should be
uncorrelated. Whatever common factor survives appears as positive correlation
between the residuals of unrelated portfolios, and as a leading eigenvalue above
the noise floor. Neither quantity mentions a hedging instrument.

Read the hedged figures against the **overlap floor**, not against zero.

In [ ]:
# ═══ CELL 5 ═══
order = ["unhedged"] + list(HEDGES) + ["overlap_null"]
rows = []
for (scheme, n), res in RESULTS.items():
    for o in order:
        rows.append(dict(scheme=scheme, n=n, hedge=o,
                         mean_corr=res["cf"][o]["mean_corr"], lam1=res["cf"][o]["lam1"]))
cf_tbl = pd.DataFrame(rows)

print("portfolios x days used, and the MP edge, per combination:")
for (scheme, n), res in RESULTS.items():
    u = res["cf"]["unhedged"]
    print(f"  {scheme:10s} n={n:3d}: K={u['K']:4d} T={u['T']:5d} MP edge={u['mp_edge']:.2f}")

for scheme in CFG.schemes:
    print(f"\n=== {scheme} ===")
    print("mean pairwise residual correlation")
    display(cf_tbl[cf_tbl.scheme == scheme].pivot(index="n", columns="hedge",
                                                  values="mean_corr")[order].round(4))
    print("leading eigenvalue")
    display(cf_tbl[cf_tbl.scheme == scheme].pivot(index="n", columns="hedge",
                                                  values="lam1")[order].round(2))

fig, axes = plt.subplots(len(CFG.schemes), len(CFG.sizes),
                         figsize=(4 * len(CFG.sizes), 3.4 * len(CFG.schemes)),
                         squeeze=False, sharey="row")
cols = ["gray", "steelblue", "darkorange", "seagreen", "black"]
for i, scheme in enumerate(CFG.schemes):
    for j, n in enumerate(CFG.sizes):
        ax = axes[i][j]
        sub = cf_tbl[(cf_tbl.scheme == scheme) & (cf_tbl.n == n)].set_index("hedge")
        ax.bar(range(len(order)), [sub.loc[o, "mean_corr"] for o in order], color=cols)
        ax.set_xticks(range(len(order)))
        ax.set_xticklabels(["unhedged", "SPY", "RSP", "PC1", "floor"],
                           rotation=45, ha="right", fontsize=8)
        ax.set_title(f"{scheme}, n={n}", fontsize=9); ax.grid(alpha=0.25, axis="y")
    axes[i][0].set_ylabel("mean residual correlation")
plt.tight_layout(); savefig("fig_residual_correlation")
plt.show()

## Cell 6 — Is PC1's advantage over RSP significant?

This is the test for **hypothesis 2**. Four point estimates pointing the same
way are suggestive but not evidence, so the difference is tested directly.

Days are resampled in blocks with the stationary bootstrap, and the **same**
resampled index is applied to both instruments so the difference stays paired.
Resampling days rather than trials is the point: the 1000 portfolios share one
history, so their spread across portfolios is not uncertainty about a mean.

Only mean correlation is bootstrapped, not the leading eigenvalue. An average
over pairs has an expectation that does not depend on how many portfolios are
sampled, so a subsample is unbiased for it; `lambda_1` grows with `K` and would
not be comparable across subsample sizes.

In [ ]:
# ═══ CELL 6 ═══
def mean_corr(X):
    X = np.asarray(X, dtype=float)
    Xc = X - X.mean(axis=0)
    sd = Xc.std(axis=0)
    ok = sd > 0
    if ok.sum() < 3:
        return np.nan
    Z = Xc[:, ok] / sd[ok]
    K = Z.shape[1]
    C = (Z.T @ Z) / Z.shape[0]
    return float((C.sum() - np.trace(C)) / (K * (K - 1)))


def bootstrap_mean_corr_diff(aligned, a, b, n_boot, mean_block, seed, alpha=0.05):
    A, B = aligned[a], aligned[b]
    T = A.shape[0]
    if T < mean_block * 2:
        return np.nan, (np.nan, np.nan)
    rng_l = np.random.default_rng(seed)
    p = 1.0 / mean_block
    out = np.empty(n_boot)
    for i in range(n_boot):
        lens, st, tot = [], [], 0
        while tot < T:
            L = int(rng_l.geometric(p)); lens.append(L); st.append(int(rng_l.integers(0, T))); tot += L
        offs = np.concatenate([np.arange(L) for L in lens])[:T]
        base = np.repeat(np.array(st), np.array(lens))[:T]
        idx = (base + offs) % T
        out[i] = mean_corr(A[idx]) - mean_corr(B[idx])
    return mean_corr(A) - mean_corr(B), tuple(np.quantile(out, [alpha / 2, 1 - alpha / 2]))


rho_rows = []
for (scheme, n), al in ALIGNED.items():
    for a, b in [("PC1", "RSP"), ("PC1", "SPY")]:
        pt, (lo, hi) = bootstrap_mean_corr_diff(al, a, b, CFG.boot_rho_n,
                                                CFG.boot_block, CFG.seed)
        verdict = (f"favours {a}" if hi < 0 else
                   f"favours {b}" if lo > 0 else "not significant")
        rho_rows.append(dict(scheme=scheme, n=n, pair=f"{a} - {b}",
                             diff=pt, lo=lo, hi=hi, verdict=verdict))

rho_tbl = pd.DataFrame(rho_rows)
print("Difference in mean residual correlation, 95% block-bootstrap CI over days")
print("(negative favours the first instrument: less common factor left behind)\n")
display(rho_tbl.set_index(["scheme", "n", "pair"]).round(5))

print("\nTally of verdicts:")
print(rho_tbl.groupby(["pair", "verdict"]).size().to_string())

## Cell 7 — Variance reduction (hypothesis 3)

The practical hedger's objective, and the conventional measure of hedging
effectiveness. Paired across instruments on the same portfolios, with the
interval again taken over days.

In [ ]:
# ═══ CELL 7 ═══
def block_bootstrap_mean_ci(x, n_boot, mean_block, seed, alpha=0.05):
    rng_l = np.random.default_rng(seed)
    x = np.asarray(pd.Series(x).dropna(), float)
    n = len(x)
    if n < mean_block * 2:
        return np.nan, np.nan
    p = 1.0 / mean_block
    out = np.empty(n_boot)
    for b in range(n_boot):
        lens, st, tot = [], [], 0
        while tot < n:
            L = int(rng_l.geometric(p)); lens.append(L); st.append(int(rng_l.integers(0, n))); tot += L
        offs = np.concatenate([np.arange(L) for L in lens])[:n]
        base = np.repeat(np.array(st), np.array(lens))[:n]
        out[b] = x[(base + offs) % n].mean()
    return tuple(np.quantile(out, [alpha / 2, 1 - alpha / 2]))


vr_rows, sq_rows = [], []
for (scheme, n), res in RESULTS.items():
    for h in HEDGES:
        vr_rows.append(dict(scheme=scheme, n=n, hedge=h, mean_VR=res["vr"][h],
                            p10=res["vr_p10"][h], p90=res["vr_p90"][h]))
    for pair, d_t in res["sq"].items():
        lo, hi = block_bootstrap_mean_ci(d_t, CFG.boot_n, CFG.boot_block, CFG.seed)
        a, b = pair.split("-")
        sq_rows.append(dict(scheme=scheme, n=n, pair=f"{a} - {b}",
                            mean_sq_diff=float(np.nanmean(d_t)), lo=lo, hi=hi,
                            verdict=(f"favours {a}" if hi < 0 else
                                     f"favours {b}" if lo > 0 else "not significant")))

print("Variance reduction  1 - Var(residual)/Var(portfolio)")
print("p10/p90 are dispersion ACROSS PORTFOLIOS, not confidence intervals\n")
for scheme in CFG.schemes:
    print(f"=== {scheme} ===")
    display(pd.DataFrame(vr_rows).query("scheme == @scheme").pivot(
        index="n", columns="hedge", values="mean_VR")[list(HEDGES)].round(4))

print("\nPaired mean squared-residual difference, 95% CI over days")
print("Use this, not VR, to judge the size sweep: it is not scale-free either,")
print("since it carries variance units that shrink with n, but it is free of")
print("VR's denominator, which falls with n for reasons unrelated to hedging.\n")
display(pd.DataFrame(sq_rows).set_index(["scheme", "n", "pair"]).round(10))

## Cell 8 — Spanning, both directions

In [ ]:
# ═══ CELL 8 ═══
sp_rows = [dict(scheme=s, n=n, test=k, mean_beta=v)
           for (s, n), res in RESULTS.items() for k, v in res["span"].items()]
print("Residual exposure left to the other instrument (0 = fully spanned)\n")
for scheme in CFG.schemes:
    print(f"=== {scheme} ===")
    display(pd.DataFrame(sp_rows).query("scheme == @scheme").pivot(
        index="n", columns="test", values="mean_beta").round(4))
print("PC1 and SPY correlate about 0.93, so a joint regression would be badly")
print("collinear (VIF ~7) and its coefficients unstable. The one-at-a-time form")
print("asks the same question without that fragility.")

## Cell 9 — Year and volatility regime

A hedge that works in calm markets and fails in crises is worthless, so the
regime cut is the one that matters. Regimes come from **ex-ante** volatility —
SPY's trailing EWMA sigma as known at `t-1`. Tercile cutoffs are computed over
the whole window; they only group days for reporting and never enter a weight or
a hedge ratio.

In [ ]:
# ═══ CELL 9 ═══
cut_rows = [dict(scheme=s, n=n, cut=label, **vals)
            for (s, n), res in RESULTS.items() for label, vals in res["cuts"].items()]
cuts = pd.DataFrame(cut_rows)

print("ex-ante vol terciles (annualised): "
      f"calm <= {q1*np.sqrt(252):.1%} < normal <= {q2*np.sqrt(252):.1%} < stressed")
print(REGIME.value_counts().to_string())

for scheme in CFG.schemes:
    print(f"\n=== {scheme}: variance reduction by regime ===")
    sub = cuts[(cuts.scheme == scheme) & (~cuts.cut.str.startswith("year"))]
    for h in HEDGES:
        print(f"  {h}")
        display(sub.pivot(index="cut", columns="n", values=h).round(3))

print("\n=== by year (PC1) ===")
display(cuts[cuts.cut.str.startswith("year")].pivot_table(
    index="cut", columns=["scheme", "n"], values="PC1").round(3))

## Cell 10 — Costs

SPY is one liquid ETF. PC1 is a ~600-name basket that rebalances itself. Both
are charged `cost_bps` per side on hedge-notional turnover, and PC1 additionally
pays its own measured annual turnover. The ETFs rebalance internally and are not
charged for it here.

In [ ]:
# ═══ CELL 10 ═══
cost_rows = [dict(scheme=s, n=n, hedge=h, ann_cost=res["cost"][h])
             for (s, n), res in RESULTS.items() for h in HEDGES]
print(f"Annualised cost of running each hedge, at {CFG.cost_bps:.0f} bps per side")
print(f"PC1 also pays {CFG.pc1_annual_turnover:.1f}x annual internal turnover\n")
for scheme in CFG.schemes:
    print(f"=== {scheme} ===")
    display(pd.DataFrame(cost_rows).query("scheme == @scheme").pivot(
        index="n", columns="hedge", values="ann_cost")[list(HEDGES)].round(5))

## Cell 11 — the three hypotheses

Stated separately, because they are different questions with different answers.

In [ ]:
# ═══ CELL 11 ═══
def tally(tbl, pair):
    sub = tbl[tbl.pair == pair]
    return sub.verdict.value_counts().to_dict(), len(sub)


print("=" * 78)
print("HYPOTHESIS 1  PC1 removes more systematic risk than SPY")
print("=" * 78)
t1r, n1r = tally(rho_tbl, "PC1 - SPY")
t1s, n1s = tally(pd.DataFrame(sq_rows), "PC1 - SPY")
print(f"  residual correlation : {t1r}  of {n1r} combinations")
print(f"  squared residuals    : {t1s}  of {n1s} combinations")
for scheme in CFG.schemes:
    vrs = [(n, RESULTS[(scheme, n)]["vr"]["PC1"], RESULTS[(scheme, n)]["vr"]["SPY"])
           for n in CFG.sizes]
    print(f"  {scheme:10s} VR PC1 vs SPY: " +
          "  ".join(f"n={n}: {a:.3f}/{b:.3f}" for n, a, b in vrs))

print("\n" + "=" * 78)
print("HYPOTHESIS 2  PC1 removes more systematic risk than RSP  (residual cross-section)")
print("=" * 78)
t2, n2 = tally(rho_tbl, "PC1 - RSP")
print(f"  verdicts: {t2}  of {n2} combinations")
display(rho_tbl[rho_tbl.pair == "PC1 - RSP"].set_index(["scheme", "n"])[
    ["diff", "lo", "hi", "verdict"]].round(5))

print("\n" + "=" * 78)
print("HYPOTHESIS 3  PC1 reduces more total variance than RSP  (squared residuals)")
print("=" * 78)
sq_tbl = pd.DataFrame(sq_rows)
t3, n3 = tally(sq_tbl, "PC1 - RSP")
print(f"  verdicts: {t3}  of {n3} combinations")
display(sq_tbl[sq_tbl.pair == "PC1 - RSP"].set_index(["scheme", "n"])[
    ["mean_sq_diff", "lo", "hi", "verdict"]].round(10))

print("\n" + "-" * 78)
print("Size sweep: PC1 - SPY squared-residual difference across sizes")
print("(flat in n rules out convergence on equal weighting as n grows;")
print(" whether the advantage IS a weighting effect is Hypothesis 2's job)")
for scheme in CFG.schemes:
    vals = [sq_tbl.query("scheme == @scheme and n == @n and pair == 'PC1 - SPY'"
                         )["mean_sq_diff"].iloc[0] for n in CFG.sizes]
    print(f"  {scheme:10s} " + "  ".join(f"n={n}: {v:+.3e}" for n, v in zip(CFG.sizes, vals)))

print("\nTerrain check: the size-tilted scheme moves the test toward SPY's own")
print("weighting. If hypothesis 1 survives there, it is not an artefact of drawing")
print("portfolios that are equal-weighted in expectation.")

## Cell 12 — Acceptance tests

In [ ]:
# ═══ CELL 12 ═══
T_RES = []
def check(name, ok, detail=""):
    T_RES.append((name, bool(ok), detail))
    print(f"  {'PASS' if ok else 'FAIL'}  {name}" + (f"   {detail}" if detail else ""))


print("[1] Samplers")
_r = np.random.default_rng(0)
for n in CFG.sizes:
    if n == 1:
        continue
    w = stick_breaking_weights(n, 20000, _r)
    H = sum(1.0 / i for i in range(1, n + 1))
    check(f"dirichlet n={n}: sums to 1", np.allclose(w.sum(axis=1), 1.0))
    check(f"dirichlet n={n}: E[max w] = H_n/n", abs(w.max(axis=1).mean() - H / n) < 0.01,
          f"{w.max(axis=1).mean():.4f} vs {H/n:.4f}")
_flat = tilted_weights(np.ones((20000, 20)), np.random.default_rng(1))
_H20 = sum(1.0 / i for i in range(1, 21))
check("tilted with a flat proxy reproduces Dirichlet(1)",
      abs(_flat.max(axis=1).mean() - _H20 / 20) < 0.005,
      f"{_flat.max(axis=1).mean():.4f} vs {_H20/20:.4f}")
check("the tilt actually concentrates weight",
      RESULTS[("tilted", 20)]["maxw"] > 1.3 * RESULTS[("dirichlet", 20)]["maxw"],
      f"E[max w] {RESULTS[('tilted',20)]['maxw']:.3f} tilted vs "
      f"{RESULTS[('dirichlet',20)]['maxw']:.3f} dirichlet")

print("\n[1b] Schemes share their draws")
# With one name per portfolio both schemes assign weight 1, so if the names are
# genuinely shared every n = 1 figure must agree EXACTLY. Any difference means
# the schemes drew different stocks and the comparison between them is
# confounded with sampling.
if 1 in CFG.sizes and len(CFG.schemes) > 1:
    a, b = CFG.schemes[0], CFG.schemes[1]
    ra, rb = RESULTS[(a, 1)], RESULTS[(b, 1)]
    check("n=1: variance reduction identical across schemes",
          all(ra["vr"][h] == rb["vr"][h] for h in HEDGES),
          "  ".join(f"{h} {ra['vr'][h]:.6f}/{rb['vr'][h]:.6f}" for h in HEDGES))
    check("n=1: residual correlation identical across schemes",
          all(ra["cf"][h]["mean_corr"] == rb["cf"][h]["mean_corr"] for h in HEDGES),
          "  ".join(f"{h} {ra['cf'][h]['mean_corr']:+.6f}" for h in HEDGES))
    check("n=1: both schemes assign unit weight",
          abs(ra["maxw"] - 1.0) < 1e-12 and abs(rb["maxw"] - 1.0) < 1e-12,
          f"max w = {ra['maxw']:.6f} / {rb['maxw']:.6f}")
    # and for n > 1 the schemes must NOT be identical, or the tilt does nothing
    big = CFG.sizes[-1]
    check(f"n={big}: schemes genuinely differ",
          RESULTS[(a, big)]["maxw"] != RESULTS[(b, big)]["maxw"],
          f"max w = {RESULTS[(a,big)]['maxw']:.4f} / {RESULTS[(b,big)]['maxw']:.4f}")

print("\n[2] Point-in-time draw")
check("every eligible name was an index member with data on the draw date",
      all(bool(Mv[0, j]) and bool(Vv[0, j]) for j in ELIGIBLE))

print("\n[3] Self-hedge")
_p = portfolio_returns(weight_matrix(draw_names(ELIGIBLE, 20, 1, np.random.default_rng(3)),
                                     stick_breaking_weights(20, 1, np.random.default_rng(3)),
                                     rets_w.shape[1]), Rv, Vv, CFG.min_surviving)[0][:, 0]
_s = pd.Series(_p, index=WIN)
_cov = pd.DataFrame({"p": _p}, index=WIN).rolling(
    CFG.beta_lookback, min_periods=CFG.beta_min_obs).cov(_s).to_numpy()
_var = _s.rolling(CFG.beta_lookback, min_periods=CFG.beta_min_obs).var().to_numpy()
_b = hedge_betas(np.ones((1, 1)), np.ones((len(WIN), 1), bool), _cov, _var,
                 RB, len(WIN), CFG.min_surviving)
check("beta on self is 1", np.nanmax(np.abs(_b - 1.0)) < 1e-8,
      f"max|beta-1| = {np.nanmax(np.abs(_b - 1.0)):.2e}")
check("residual on self is 0", np.nanmax(np.abs(_p[:, None] - _b * _p[:, None])) < 1e-12)

print("\n[4] Future invariance")
CUT = len(WIN) // 2
_rng = np.random.default_rng(CFG.seed)
rets_p = rets_w.copy()
rets_p.iloc[CUT:] = np.where(Vv[CUT:],
                             _rng.normal(0, 0.35, (len(WIN) - CUT, rets_w.shape[1])), np.nan)
hedge_p = hedge_w.copy()
hedge_p.iloc[CUT:] = _rng.normal(0, 0.35, (len(WIN) - CUT, len(HEDGES)))
_pk = draw_names(ELIGIBLE, 20, 200, np.random.default_rng(9))
_W = weight_matrix(_pk, stick_breaking_weights(20, 200, np.random.default_rng(9)),
                   rets_w.shape[1])
# All three instruments, and through the cutoff row itself: the beta in
# force on day t comes from the last rebalance at or before t and uses
# moments through tau-1, so nothing on that row may move when the future is
# replaced by noise. Checking only the rows strictly before it would leave
# room for a ratio that read the current day.
_ok_fi, _det = True, []
for _h in HEDGES:
    _b0 = hedge_betas(_W, Vv, COV[_h], VAR[_h], RB, len(WIN), CFG.min_surviving)
    _cp = rets_p.rolling(CFG.beta_lookback,
                         min_periods=CFG.beta_min_obs).cov(hedge_p[_h]).to_numpy()
    _vp = hedge_p[_h].rolling(CFG.beta_lookback,
                              min_periods=CFG.beta_min_obs).var().to_numpy()
    _b1 = hedge_betas(_W, Vv, _cp, _vp, RB, len(WIN), CFG.min_surviving)
    _same = np.allclose(_b0[:CUT + 1], _b1[:CUT + 1], equal_nan=True,
                        rtol=0, atol=0)
    _ok_fi &= _same
    _det.append(f"{_h} {'ok' if _same else 'DIFFERS'}")
check("betas through the cutoff row are bit-identical, all instruments", _ok_fi,
      f"cutoff {WIN[CUT]:%Y-%m-%d}, " + ", ".join(_det))

print("\n[5] The metric is calibrated")
for scheme in CFG.schemes:
    cf = RESULTS[(scheme, 20)]["cf"]
    check(f"{scheme}: unhedged factor far exceeds the overlap floor",
          cf["unhedged"]["lam1"] > 3 * cf["overlap_null"]["lam1"],
          f"{cf['unhedged']['lam1']:.1f} vs {cf['overlap_null']['lam1']:.1f}")
    check(f"{scheme}: every hedge reduces the residual spike",
          all(cf[h]["lam1"] < cf["unhedged"]["lam1"] for h in HEDGES))
check("paired panels share one grid",
      len({A.shape for A in ALIGNED[("dirichlet", 20)].values()}) == 1,
      str({h: A.shape for h, A in ALIGNED[("dirichlet", 20)].items()}))
_al0 = ALIGNED[("dirichlet", 20)]
_pt0, (_lo0, _hi0) = bootstrap_mean_corr_diff({"x": _al0["PC1"], "y": _al0["PC1"]},
                                              "x", "y", 50, CFG.boot_block, CFG.seed)
check("a panel against itself gives exactly zero difference", abs(_pt0) < 1e-12,
      f"{_pt0:+.2e}")

print("\n[6] Synthetic recovery")
_T, _N, _K = 900, 300, 120
_g = np.random.default_rng(CFG.seed)
_m = _g.normal(0, 0.01, _T)
_bta = np.clip(_g.normal(1, .25, _N), .3, None)
_R = _m[:, None] * _bta[None, :] + _g.normal(0, 1, (_T, _N)) * _g.uniform(.008, .02, _N)
_V = np.ones((_T, _N), bool)
_pk2 = draw_names(np.arange(_N), 20, _K, np.random.default_rng(1))
_Wm = weight_matrix(_pk2, stick_breaking_weights(20, _K, np.random.default_rng(1)), _N)
_rp, _ = portfolio_returns(_Wm, _R, _V, CFG.min_surviving)
_df = pd.DataFrame(_R, index=pd.bdate_range("2015-01-01", periods=_T))
_ms = pd.Series(_m, index=_df.index)
_cv = _df.rolling(CFG.beta_lookback, min_periods=CFG.beta_min_obs).cov(_ms).to_numpy()
_vr2 = _ms.rolling(CFG.beta_lookback, min_periods=CFG.beta_min_obs).var().to_numpy()
_rb = np.arange(CFG.beta_lookback, _T, CFG.rebalance_days)
_bb = hedge_betas(_Wm, _V, _cv, _vr2, _rb, _T, CFG.min_surviving)
_ee = _rp - _bb * _m[:, None]
_un = residual_common_factor(_rp, _K, CFG.seed, CFG.panel_min_cover)
_hd = residual_common_factor(_ee, _K, CFG.seed, CFG.panel_min_cover)
check("planted factor detected unhedged", _un["lam1"] > 20 * _un["mp_edge"],
      f"lam1 = {_un['lam1']:.1f}")
check("hedging the true factor collapses the spike", _hd["lam1"] < 0.15 * _un["lam1"],
      f"{_un['lam1']:.1f} -> {_hd['lam1']:.1f}")

# Hedging against a series with nothing to do with the panel must not
# manufacture exposure: the ratio should sit at zero and the hedge should
# remove no variance. Without this, a hedge that merely added noise of the
# right scale could look as though it were working.
_un_rel = _g.normal(0, 0.01, _T)
_cv_u = _df.rolling(CFG.beta_lookback, min_periods=CFG.beta_min_obs).cov(
    pd.Series(_un_rel, index=_df.index)).to_numpy()
_vr_u = pd.Series(_un_rel, index=_df.index).rolling(
    CFG.beta_lookback, min_periods=CFG.beta_min_obs).var().to_numpy()
_b_u = hedge_betas(_Wm, _V, _cv_u, _vr_u, _rb, _T, CFG.min_surviving)
_e_u = _rp - _b_u * _un_rel[:, None]
_vr_unrel = float(np.nanmean(variance_reduction(_rp, _e_u)))
check("hedging an unrelated series moves nothing",
      abs(float(np.nanmean(_b_u))) < 0.05 and abs(_vr_unrel) < 0.02,
      f"mean beta {np.nanmean(_b_u):+.4f}, variance reduction {_vr_unrel:+.4f}")

# With no common factor at all, the residual cross-section should sit at the
# overlap floor rather than at zero: what is left is the correlation shared
# holdings induce, and nothing else. This is the calibration the whole metric
# rests on.
_Rn = _g.normal(0, 1, (_T, _N)) * _g.uniform(.008, .02, _N)   # idiosyncratic only
_rn, _ = portfolio_returns(_Wm, _Rn, _V, CFG.min_surviving)
_none = residual_common_factor(_rn, _K, CFG.seed, CFG.panel_min_cover)
_floor = overlap_null(_Wm, _Rn, _V, CFG.seed, _K, CFG.min_surviving,
                      CFG.panel_min_cover)
check("with no common factor the panel sits at the overlap floor",
      abs(_none["lam1"] - _floor["lam1"]) < 0.25 * _floor["lam1"],
      f"lam1 {_none['lam1']:.2f} vs floor {_floor['lam1']:.2f}")

_f = [n for n, ok, _ in T_RES if not ok]
print("\n" + "=" * 70)
print(f"{len(T_RES)-len(_f)}/{len(T_RES)} passed")
for n in _f:
    print(f"  FAILED: {n}")
print("=" * 70)
assert not _f, f"acceptance tests failed: {_f}"
print("PASS - lag-clean, calibrated metric, both samplers behaving.")